# Ngày 10: Đánh giá một mô hình có đáng tin không?

hiểu train/validation/test và so sánh MAE, MSE, R² với baseline dự đoán trung bình.

A · So baseline trung bình với mô hình; giải thích từng phần train/validation/test.

B · Tính [MAE,MSE,R²] cho truth và pred dưới đây.

C (tùy chọn) · Thêm một ngoại lệ vào dự đoán; giải thích chỉ số nào biến động mạnh.

Chạy tất cả để nạp hàm hỗ trợ và xem ví dụ. Ô bài B còn answer=None sẽ báo CHƯA HOÀN THÀNH. Điền mã rồi chạy lại; ĐẠT là kiểm tra kết quả bài B, không thay thế việc giải thích và nhật ký. Ví dụ mô phỏng hoặc bảng giả lập được ghi rõ. Iris: R. Fisher, UCI, DOI 10.24432/C56C76, CC BY 4.0, 150 dòng phiên bản legacy.

## Đọc trước khi tự thực hành

Câu hỏi: Mô hình sai bao nhiêu thì được coi là hữu ích?

Bản chất: MAE=mean|error| cùng đơn vị y; RMSE=√MSE cũng cùng đơn vị; R² so với biến thiên y trên tập đang đánh giá và có thể âm. So với baseline trước khi khen mô hình.

Ví dụ tính tay: y=[1,3], ŷ=[2,2]: MAE=1, MSE=1, R²=0. Một baseline hằng dễ hiểu là mean train; median train là hằng tối ưu cho MAE trên train.

Thử và quan sát: Tính metric tay trên vài mẫu, rồi xem train/validation. Viết sai số này có chấp nhận được cho mục đích sử dụng không.

Tự kiểm tra: Em báo cáo metric kèm đơn vị, tập đo và baseline.

Dễ nhầm: Không diễn giải MSE như tiền sai trung bình, vì đơn vị đã bình phương.

Đọc Cam_nang_tu_hoc_AI_30_ngay.md khi cần tra cứu; giữ answer=None cho tới khi em tự thử.

In [ ]:
IRIS_CSV = "sepal_length,sepal_width,petal_length,petal_width,species\n5.1,3.5,1.4,0.2,Iris-setosa\n4.9,3.0,1.4,0.2,Iris-setosa\n4.7,3.2,1.3,0.2,Iris-setosa\n4.6,3.1,1.5,0.2,Iris-setosa\n5.0,3.6,1.4,0.2,Iris-setosa\n5.4,3.9,1.7,0.4,Iris-setosa\n4.6,3.4,1.4,0.3,Iris-setosa\n5.0,3.4,1.5,0.2,Iris-setosa\n4.4,2.9,1.4,0.2,Iris-setosa\n4.9,3.1,1.5,0.1,Iris-setosa\n5.4,3.7,1.5,0.2,Iris-setosa\n4.8,3.4,1.6,0.2,Iris-setosa\n4.8,3.0,1.4,0.1,Iris-setosa\n4.3,3.0,1.1,0.1,Iris-setosa\n5.8,4.0,1.2,0.2,Iris-setosa\n5.7,4.4,1.5,0.4,Iris-setosa\n5.4,3.9,1.3,0.4,Iris-setosa\n5.1,3.5,1.4,0.3,Iris-setosa\n5.7,3.8,1.7,0.3,Iris-setosa\n5.1,3.8,1.5,0.3,Iris-setosa\n5.4,3.4,1.7,0.2,Iris-setosa\n5.1,3.7,1.5,0.4,Iris-setosa\n4.6,3.6,1.0,0.2,Iris-setosa\n5.1,3.3,1.7,0.5,Iris-setosa\n4.8,3.4,1.9,0.2,Iris-setosa\n5.0,3.0,1.6,0.2,Iris-setosa\n5.0,3.4,1.6,0.4,Iris-setosa\n5.2,3.5,1.5,0.2,Iris-setosa\n5.2,3.4,1.4,0.2,Iris-setosa\n4.7,3.2,1.6,0.2,Iris-setosa\n4.8,3.1,1.6,0.2,Iris-setosa\n5.4,3.4,1.5,0.4,Iris-setosa\n5.2,4.1,1.5,0.1,Iris-setosa\n5.5,4.2,1.4,0.2,Iris-setosa\n4.9,3.1,1.5,0.1,Iris-setosa\n5.0,3.2,1.2,0.2,Iris-setosa\n5.5,3.5,1.3,0.2,Iris-setosa\n4.9,3.1,1.5,0.1,Iris-setosa\n4.4,3.0,1.3,0.2,Iris-setosa\n5.1,3.4,1.5,0.2,Iris-setosa\n5.0,3.5,1.3,0.3,Iris-setosa\n4.5,2.3,1.3,0.3,Iris-setosa\n4.4,3.2,1.3,0.2,Iris-setosa\n5.0,3.5,1.6,0.6,Iris-setosa\n5.1,3.8,1.9,0.4,Iris-setosa\n4.8,3.0,1.4,0.3,Iris-setosa\n5.1,3.8,1.6,0.2,Iris-setosa\n4.6,3.2,1.4,0.2,Iris-setosa\n5.3,3.7,1.5,0.2,Iris-setosa\n5.0,3.3,1.4,0.2,Iris-setosa\n7.0,3.2,4.7,1.4,Iris-versicolor\n6.4,3.2,4.5,1.5,Iris-versicolor\n6.9,3.1,4.9,1.5,Iris-versicolor\n5.5,2.3,4.0,1.3,Iris-versicolor\n6.5,2.8,4.6,1.5,Iris-versicolor\n5.7,2.8,4.5,1.3,Iris-versicolor\n6.3,3.3,4.7,1.6,Iris-versicolor\n4.9,2.4,3.3,1.0,Iris-versicolor\n6.6,2.9,4.6,1.3,Iris-versicolor\n5.2,2.7,3.9,1.4,Iris-versicolor\n5.0,2.0,3.5,1.0,Iris-versicolor\n5.9,3.0,4.2,1.5,Iris-versicolor\n6.0,2.2,4.0,1.0,Iris-versicolor\n6.1,2.9,4.7,1.4,Iris-versicolor\n5.6,2.9,3.6,1.3,Iris-versicolor\n6.7,3.1,4.4,1.4,Iris-versicolor\n5.6,3.0,4.5,1.5,Iris-versicolor\n5.8,2.7,4.1,1.0,Iris-versicolor\n6.2,2.2,4.5,1.5,Iris-versicolor\n5.6,2.5,3.9,1.1,Iris-versicolor\n5.9,3.2,4.8,1.8,Iris-versicolor\n6.1,2.8,4.0,1.3,Iris-versicolor\n6.3,2.5,4.9,1.5,Iris-versicolor\n6.1,2.8,4.7,1.2,Iris-versicolor\n6.4,2.9,4.3,1.3,Iris-versicolor\n6.6,3.0,4.4,1.4,Iris-versicolor\n6.8,2.8,4.8,1.4,Iris-versicolor\n6.7,3.0,5.0,1.7,Iris-versicolor\n6.0,2.9,4.5,1.5,Iris-versicolor\n5.7,2.6,3.5,1.0,Iris-versicolor\n5.5,2.4,3.8,1.1,Iris-versicolor\n5.5,2.4,3.7,1.0,Iris-versicolor\n5.8,2.7,3.9,1.2,Iris-versicolor\n6.0,2.7,5.1,1.6,Iris-versicolor\n5.4,3.0,4.5,1.5,Iris-versicolor\n6.0,3.4,4.5,1.6,Iris-versicolor\n6.7,3.1,4.7,1.5,Iris-versicolor\n6.3,2.3,4.4,1.3,Iris-versicolor\n5.6,3.0,4.1,1.3,Iris-versicolor\n5.5,2.5,4.0,1.3,Iris-versicolor\n5.5,2.6,4.4,1.2,Iris-versicolor\n6.1,3.0,4.6,1.4,Iris-versicolor\n5.8,2.6,4.0,1.2,Iris-versicolor\n5.0,2.3,3.3,1.0,Iris-versicolor\n5.6,2.7,4.2,1.3,Iris-versicolor\n5.7,3.0,4.2,1.2,Iris-versicolor\n5.7,2.9,4.2,1.3,Iris-versicolor\n6.2,2.9,4.3,1.3,Iris-versicolor\n5.1,2.5,3.0,1.1,Iris-versicolor\n5.7,2.8,4.1,1.3,Iris-versicolor\n6.3,3.3,6.0,2.5,Iris-virginica\n5.8,2.7,5.1,1.9,Iris-virginica\n7.1,3.0,5.9,2.1,Iris-virginica\n6.3,2.9,5.6,1.8,Iris-virginica\n6.5,3.0,5.8,2.2,Iris-virginica\n7.6,3.0,6.6,2.1,Iris-virginica\n4.9,2.5,4.5,1.7,Iris-virginica\n7.3,2.9,6.3,1.8,Iris-virginica\n6.7,2.5,5.8,1.8,Iris-virginica\n7.2,3.6,6.1,2.5,Iris-virginica\n6.5,3.2,5.1,2.0,Iris-virginica\n6.4,2.7,5.3,1.9,Iris-virginica\n6.8,3.0,5.5,2.1,Iris-virginica\n5.7,2.5,5.0,2.0,Iris-virginica\n5.8,2.8,5.1,2.4,Iris-virginica\n6.4,3.2,5.3,2.3,Iris-virginica\n6.5,3.0,5.5,1.8,Iris-virginica\n7.7,3.8,6.7,2.2,Iris-virginica\n7.7,2.6,6.9,2.3,Iris-virginica\n6.0,2.2,5.0,1.5,Iris-virginica\n6.9,3.2,5.7,2.3,Iris-virginica\n5.6,2.8,4.9,2.0,Iris-virginica\n7.7,2.8,6.7,2.0,Iris-virginica\n6.3,2.7,4.9,1.8,Iris-virginica\n6.7,3.3,5.7,2.1,Iris-virginica\n7.2,3.2,6.0,1.8,Iris-virginica\n6.2,2.8,4.8,1.8,Iris-virginica\n6.1,3.0,4.9,1.8,Iris-virginica\n6.4,2.8,5.6,2.1,Iris-virginica\n7.2,3.0,5.8,1.6,Iris-virginica\n7.4,2.8,6.1,1.9,Iris-virginica\n7.9,3.8,6.4,2.0,Iris-virginica\n6.4,2.8,5.6,2.2,Iris-virginica\n6.3,2.8,5.1,1.5,Iris-virginica\n6.1,2.6,5.6,1.4,Iris-virginica\n7.7,3.0,6.1,2.3,Iris-virginica\n6.3,3.4,5.6,2.4,Iris-virginica\n6.4,3.1,5.5,1.8,Iris-virginica\n6.0,3.0,4.8,1.8,Iris-virginica\n6.9,3.1,5.4,2.1,Iris-virginica\n6.7,3.1,5.6,2.4,Iris-virginica\n6.9,3.1,5.1,2.3,Iris-virginica\n5.8,2.7,5.1,1.9,Iris-virginica\n6.8,3.2,5.9,2.3,Iris-virginica\n6.7,3.3,5.7,2.5,Iris-virginica\n6.7,3.0,5.2,2.3,Iris-virginica\n6.3,2.5,5.0,1.9,Iris-virginica\n6.5,3.0,5.2,2.0,Iris-virginica\n6.2,3.4,5.4,2.3,Iris-virginica\n5.9,3.0,5.1,1.8,Iris-virginica\n"
# Shared teaching helpers. Small CPU-only datasets; NumPy + Matplotlib.
import csv, io, math
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 105, 'font.size': 10, 'axes.grid': True, 'grid.alpha': .2})
np.set_printoptions(precision=4, suppress=True)

def softmax(x, axis=-1, temperature=1.):
    z=np.asarray(x,dtype=float)/temperature
    z=z-np.max(z,axis=axis,keepdims=True)
    p=np.exp(z)
    return p/p.sum(axis=axis,keepdims=True)

def sigmoid(x):
    return 1/(1+np.exp(-np.clip(x,-40,40)))

def mse(y,p): return float(np.mean((np.asarray(p)-y)**2))
def accuracy(y,p): return float(np.mean(np.asarray(y)==p))
def gini(y):
    _,c=np.unique(y,return_counts=True)
    return 1-float(np.sum((c/len(y))**2))
def cosine(a,b):
    a,b=np.asarray(a),np.asarray(b)
    return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))

def study_data():
    i=np.arange(24); x=np.round(.6+i*.36,2)
    y=np.round(21+7.5*x+np.sin(i*2.9)*7+np.cos(i*1.3)*3,1)
    return x[:,None],y

def split_indices(n,ratio=.8,seed=2026):
    order=list(range(n)); s=seed
    for i in range(n-1,0,-1):
        s=(1664525*s+1013904223)&0xffffffff
        j=int(s/4294967296*(i+1));order[i],order[j]=order[j],order[i]
    cut=max(2,min(n-2,int(n*ratio)))
    return np.array(order[:cut]),np.array(order[cut:])

def design_matrix(X,train,normalize=True):
    mu=X[train].mean(0) if normalize else np.zeros(X.shape[1])
    sd=X[train].std(0) if normalize else np.ones(X.shape[1])
    sd=np.where(sd>0,sd,1)
    return np.column_stack([np.ones(len(X)),(X-mu)/sd]),mu,sd

def fit(X,y,alpha=.05,epochs=400,normalize=True,ratio=.8):
    train,test=split_indices(len(y),ratio)
    Z,mu,sd=design_matrix(X,train,normalize)
    theta=np.zeros(Z.shape[1]);h=[theta.copy()]
    initial=mse(y[train],Z[train]@theta)
    for _ in range(epochs):
        grad=2/len(train)*Z[train].T@(Z[train]@theta-y[train])
        new=theta-alpha*grad
        loss=mse(y[train],Z[train]@new)
        if not np.isfinite(loss) or loss>max(initial,1)*1e10: break
        theta=new;h.append(theta.copy())
    h=np.array(h)
    return dict(theta=theta,history=h,Z=Z,mu=mu,sd=sd,train=train,test=test,
                train_mse=np.mean((Z[train]@h.T-y[train,None])**2,axis=0),
                test_mse=np.mean((Z[test]@h.T-y[test,None])**2,axis=0))

def raw_coefficients(theta,mu,sd):
    w=theta[1:]/sd
    return np.r_[theta[0]-w@mu,w]

def plot_regression(X,y,model):
    order=np.argsort(X[:,0]); fig,ax=plt.subplots(figsize=(6,3.2))
    ax.scatter(X[:,0],y,c='#238b83',label='Observed')
    ax.plot(X[order,0],(model['Z']@model['theta'])[order],c='#c4825a',label='Predicted')
    ax.set(xlabel='x1',ylabel='y');ax.legend();plt.show()

def iris_data():
    rows=list(csv.DictReader(io.StringIO(IRIS_CSV)))
    names=['sepal_length','sepal_width','petal_length','petal_width']
    X=np.array([[float(r[k]) for k in names] for r in rows])
    y=np.array([{'Iris-setosa':0,'Iris-versicolor':1,'Iris-virginica':2}[r['species']] for r in rows])
    return X,y,names

def iris_split(y):
    # The same stratified partitions are used throughout the project.
    rng=np.random.default_rng(2026);tr=[];va=[];te=[]
    for c in np.unique(y):
        idx=rng.permutation(np.flatnonzero(y==c))
        tr.extend(idx[:30]);va.extend(idx[30:40]);te.extend(idx[40:])
    return np.array(tr),np.array(va),np.array(te)

def standardize_train(X,train):
    mu=X[train].mean(0);sd=X[train].std(0);sd=np.where(sd>0,sd,1)
    return (X-mu)/sd,mu,sd

def knn_predict(X,y,Q,k=5):
    dist=np.sum((np.asarray(Q)[:,None,:]-np.asarray(X)[None,:,:])**2,axis=2)
    idx=np.argsort(dist,axis=1)[:,:k]
    return np.array([np.bincount(y[n].astype(int)).argmax() for n in idx])

def train_tree(X,y,depth=3):
    node={'label':int(np.bincount(y.astype(int)).argmax())}
    if depth==0 or len(np.unique(y))==1: return node
    best=None
    for j in range(X.shape[1]):
        values=np.unique(X[:,j]);thresholds=(values[:-1]+values[1:])/2
        for t in thresholds:
            mask=X[:,j]<=t
            if mask.sum()<2 or (~mask).sum()<2:continue
            cost=mask.mean()*gini(y[mask])+(~mask).mean()*gini(y[~mask])
            if best is None or cost<best[0]:best=(cost,j,t,mask)
    if best is None:return node
    _,j,t,mask=best
    node.update(feature=j,threshold=float(t),left=train_tree(X[mask],y[mask],depth-1),right=train_tree(X[~mask],y[~mask],depth-1))
    return node

def tree_predict(tree,Q):
    def one(x,node):
        if 'feature' not in node:return node['label']
        return one(x,node['left'] if x[node['feature']]<=node['threshold'] else node['right'])
    return np.array([one(x,tree) for x in Q])

def confusion(y,p,classes=None):
    classes=np.unique(y) if classes is None else np.asarray(classes)
    return np.array([[np.sum((y==a)&(p==b)) for b in classes] for a in classes])

def macro_f1(y,p):
    values=[]
    for c in np.unique(y):
        tp=np.sum((y==c)&(p==c));fp=np.sum((y!=c)&(p==c));fn=np.sum((y==c)&(p!=c))
        values.append(2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0)
    return float(np.mean(values))

def logistic_fit(X,y,epochs=400,lr=.15):
    mu=X.mean(0);sd=np.where(X.std(0)>0,X.std(0),1)
    Z=np.column_stack([np.ones(len(X)),(X-mu)/sd]);w=np.zeros(Z.shape[1]);loss=[]
    for _ in range(epochs):
        p=sigmoid(Z@w)
        loss.append(-np.mean(y*np.log(p+1e-10)+(1-y)*np.log(1-p+1e-10)))
        w-=lr/len(y)*Z.T@(p-y)
    return {'w':w,'mu':mu,'sd':sd,'loss':np.array(loss)}

def logistic_predict(model,X):
    Z=np.column_stack([np.ones(len(X)),(X-model['mu'])/model['sd']])
    return sigmoid(Z@model['w'])

def moons(n=120,seed=5):
    rng=np.random.default_rng(seed);t=np.linspace(0,np.pi,n//2)
    X=np.vstack([np.c_[np.cos(t),np.sin(t)],np.c_[1-np.cos(t),.5-np.sin(t)]])
    return X+rng.normal(0,.08,X.shape),np.r_[np.zeros(n//2),np.ones(n//2)].astype(int)

def train_mlp(X,y,hidden=6,epochs=650,lr=.12,seed=2):
    rng=np.random.default_rng(seed);W=rng.normal(0,.6,(X.shape[1],hidden));b=np.zeros(hidden)
    V=rng.normal(0,.6,(hidden,1));c=np.zeros(1);loss=[]
    for _ in range(epochs):
        H=np.tanh(X@W+b);p=sigmoid(H@V+c).ravel()
        loss.append(-np.mean(y*np.log(p+1e-10)+(1-y)*np.log(1-p+1e-10)))
        delta=(p-y)[:,None]/len(y);gV=H.T@delta;gc=delta.sum(0)
        dh=(delta@V.T)*(1-H**2);gW=X.T@dh;gb=dh.sum(0)
        W-=lr*gW;b-=lr*gb;V-=lr*gV;c-=lr*gc
    return dict(W=W,b=b,V=V,c=c,loss=np.array(loss))

def mlp_predict(model,X):
    return sigmoid(np.tanh(X@model['W']+model['b'])@model['V']+model['c']).ravel()

def plot_boundary(X,y,predict,title):
    a=np.linspace(X[:,0].min()-.3,X[:,0].max()+.3,70)
    b=np.linspace(X[:,1].min()-.3,X[:,1].max()+.3,70)
    xx,yy=np.meshgrid(a,b);zz=np.asarray(predict(np.c_[xx.ravel(),yy.ravel()])).reshape(xx.shape)
    fig,ax=plt.subplots(figsize=(6,3.8));ax.contourf(xx,yy,zz,levels=10,alpha=.35,cmap='cool')
    ax.scatter(X[:,0],X[:,1],c=y,cmap='cool',s=15,edgecolor='white',linewidth=.3)
    ax.set(title=title,xlabel='x1',ylabel='x2');plt.show()

def kmeans(X,k=3,steps=15,seed=1):
    rng=np.random.default_rng(seed);centers=X[rng.choice(len(X),k,replace=False)].copy();hist=[]
    for _ in range(steps):
        labels=((X[:,None,:]-centers[None,:,:])**2).sum(2).argmin(1)
        centers=np.array([X[labels==j].mean(0) if np.any(labels==j) else centers[j] for j in range(k)])
        hist.append(centers.copy())
    labels=((X[:,None,:]-centers[None,:,:])**2).sum(2).argmin(1)
    return centers,labels,hist

def pca_project(X,k=2):
    mu=X.mean(0);_,s,V=np.linalg.svd(X-mu,full_matrices=False)
    return (X-mu)@V[:k].T,V[:k],mu,s*s/(s*s).sum()

def conv_valid(image,kernel):
    h,w=image.shape;kh,kw=kernel.shape
    return np.array([[np.sum(image[i:i+kh,j:j+kw]*kernel) for j in range(w-kw+1)] for i in range(h-kh+1)])

def bandit(epsilon=.1,steps=300,seed=1):
    rng=np.random.default_rng(seed);means=np.array([.2,.5,.8]);Q=np.zeros(3);counts=np.zeros(3,int);rewards=[]
    for _ in range(steps):
        a=int(rng.integers(3)) if rng.random()<epsilon else int(Q.argmax())
        r=float(rng.random()<means[a]);counts[a]+=1;Q[a]+=(r-Q[a])/counts[a];rewards.append(r)
    return np.cumsum(rewards)/np.arange(1,steps+1),Q,counts

print('Dữ liệu và các hàm mẫu đã sẵn sàng. Đọc mã của hàm khi cần hiểu cách tính.')


## Quan sát ví dụ mẫu

So baseline trung bình với mô hình; giải thích từng phần train/validation/test.

In [ ]:
truth=np.array([1.,2.,3.]);pred=np.array([1.,2.,2.])
print('Truth:',truth,'Pred:',pred)
plt.scatter(truth,pred);plt.plot([1,3],[1,3],'--');plt.xlabel('True');plt.ylabel('Predicted');plt.show()

## Bài B: em tự làm

Tính [MAE,MSE,R²] cho truth và pred dưới đây.

Gợi ý: R² dùng tổng bình phương sai số và độ biến động quanh trung bình của truth.

In [ ]:
# Tính [MAE,MSE,R²] cho truth và pred dưới đây.
# Hoàn thành answer; dùng Gợi ý nếu vướng.
answer = None

if answer is None:
    print('CHƯA HOÀN THÀNH: điền answer rồi chạy lại ô này.')
else:
    assert np.allclose(answer,[1/3,1/3,.5])
    print('ĐẠT: kết quả bài B khớp kiểm tra. Hãy tự giải thích cách tính.')

## Thử thách C (tùy chọn)

Thêm một ngoại lệ vào dự đoán; giải thích chỉ số nào biến động mạnh.

Nhật ký: em đổi gì, quan sát gì, giải thích ra sao, một điều còn chưa rõ.